# 06 — Uncertainty Calibration (Phase 5)

Are Kriging uncertainty estimates trustworthy?

**Analysis:** Reliability diagrams (nominal vs empirical coverage) and Spearman rank correlation.

**Source:** `outputs/phase4/calibration.csv`, `outputs/phase4/spearman.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
P4 = ROOT / 'outputs' / 'phase4'
cal = pd.read_csv(P4 / 'calibration.csv')
sp = pd.read_csv(P4 / 'spearman.csv')

## Reliability Diagram (v_out_mean)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration', alpha=0.5)

for model in ['GP-M52', 'GP-SE', 'GP-SE-noARD']:
    sub = cal[(cal['model'] == model) & (cal['output'] == 'v_out_mean')]
    ax.plot(sub['nominal'], sub['empirical'], '-o', label=model, markersize=8)

ax.set_xlabel('Nominal coverage')
ax.set_ylabel('Empirical coverage')
ax.set_title('Reliability Diagram — v_out_mean')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.4, 1.0)
ax.set_ylim(0.4, 1.0)
plt.tight_layout()
plt.show()

## Calibration Table

In [ ]:
vout_cal = cal[cal['output'] == 'v_out_mean']
print('Calibration — v_out_mean:')
print(vout_cal.pivot_table(index='model', columns='nominal', values='empirical').to_string())

## Spearman Rank Correlation (all outputs)

In [ ]:
outputs = ['v_out_mean', 'i_l_mean', 'v_out_ripple', 'i_l_ripple', 'efficiency']
pivot = sp.pivot_table(index='model', columns='output', values='spearman_rho')[outputs]
print('Spearman rho (sigma_pred vs |error|):')
print(pivot.to_string())
print()
print('All correlations are positive and statistically significant (p < 1e-10).')

## Conclusion

- **GP-M52** is best calibrated: 80% nominal -> 79.9% empirical (near-perfect)
- **GP-SE / GP-SE-noARD** are overconfident (50% nominal -> 45.9% empirical)
- All models show positive Spearman rho, confirming uncertainty is **informative**
- GP-M52 rho=0.611 on v_out_mean — strongest uncertainty-error correlation